# ခရီးသွားဖောက်သည် စိတ်တိုင်းကျ ဝန်ဆောင်မှုနှင့် Handoff စီမံခန့်ခွဲမှု

ဤနိုတ်ဘုတ်သည် Microsoft Agent Framework ကို အသုံးပြုပြီး **handoff စီမံခန့်ခွဲမှု** ကို ပြသသည်။ ဖောက်သည္၏လိုအပ်ချက်အပေါ်မူတည်၍ အေးဂျင့်များသည် ကျွမ်းကျင်သူများထံအာဏာပေးပို့နိုင်သည့် ခရီးသွားဖောက်သည်စိတ်တိုင်းကျ ဝန်ဆောင်မှု စနစ်တစ်ခုကို တည်ဆောက်မည်။

## သင်ယူမည့်အရာများ
1. **Handoff စီမံခန့်ခွဲမှု**: အကြောင်းအရာနှင့် ကျွမ်းကျင်မှုအပေါ် မူတည်၍ အေးဂျင့်များကို လမ်းညွှန်ခြင်း
2. **HandoffBuilder**: Handoff workflow များ ရေးဆွဲရန် အဆင့်မြင့် API
3. **ကျွမ်းကျင်သူအသွားလမ်းညွှန်ခြင်း**: အေးဂျင့်များသည် အခြားအေးဂျင့်များထံ အလိုအလျောက်အာဏာပေးပို့နိုင်ခြင်း
4. **ပြန်လည်ဆက်သွယ်ခြင်းများ**: Handoff များအတွင်း မရှိမဖြစ်လိုအပ်သော အကြောင်းအရာ သိမ်းဆည်းမှု
5. **ဖောက်သည်စိတ်တိုင်းကျ ဝန်ဆောင်မှုလည်ပတ်မှု**: အေးဂျင့် handoff များ၏ လက်တွေ့အသုံးချမှု


## မတိုင်မီ လိုအပ်ချက်များ:
- Microsoft Agent Framework တပ်ဆင်ပြီးဖြစ်ရမည်
- GitHub token သို့မဟုတ် OpenAI API key ကို ပြင်ဆင်ထားရမည်
- အခြေခံအေးဂျင့် အယူအဆများကို နားလည်မှုရှိရမည်


In [ ]:
import asyncio
import json
import os
from collections.abc import AsyncIterable
from typing import Any, cast

from agent_framework import (
    Message,
    WorkflowEvent,
    WorkflowRunState,
)

# HandoffBuilder and the handoff request type live in the orchestrations package.
from agent_framework.orchestrations import HandoffBuilder, HandoffAgentUserRequest
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

## အဆင့် ၁: ဖော်ပြထားသော output များအတွက် Pydantic မော်ဒယ်များ သတ်မှတ်ပါ

ဒီမော်ဒယ်တွေက အထူးပြုပေးထားတဲ့ agent ထဲက တစ်ယောက်ယောက်စီ ပြန်ပေးမယ့် schema ကို သတ်မှတ်ပေးပါတယ်။ ဒီဟာက agent တွေ အကုန်မှ နည်းနည်းတူညီပြီး သော့ချက်ရလွယ်တဲ့ ဖြေကြားချက်တွေ ဖြစ်စေဖို့ ဖြစ်ပါတယ်။


In [ ]:
class FlightBookingResult(BaseModel):
    """Flight booking confirmation from the booking agent."""

    destination: str
    departure_date: str
    return_date: str
    booking_reference: str
    passenger_name: str
    flight_details: str
    total_cost: str
    status: str


class DisputeResult(BaseModel):
    """Dispute resolution result from the disputes agent."""

    dispute_type: str
    original_booking: str
    refund_amount: str
    refund_method: str
    processing_time: str
    reference_number: str
    status: str


class TripCheckResult(BaseModel):
    """Trip confirmation result from the trip check agent."""

    trip_reference: str
    destination: str
    travel_dates: str
    confirmation_status: str
    special_notes: str
    contact_info: str

## အဆင့် ၂ - ပတ်ဝန်းကျင် ဗိုလ်မှူးများအား သိမ်းဆည်းရန်


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("Microsoft Foundry provider configured successfully!")


## အဆင့် ၃: ခရီးသွားထောက်ခံမှုအတွက် အထူးပြုထားသော ဒုတိယအရာရပ် ၄ ဦးကို ဖန်တီးပါ

သတ်မှတ်ရည်မှန်းချက် တစ်ခုချင်းစီမှာ အထူးကျွမ်းကျင်မှု ရှိပြီး ဖောက်သည်လိုအပ်ချက်အရ သင့်တော်သော အထူးကျွမ်းကျင်သူများထံ လွှဲပေးနိုင်သည်။


In [ ]:
# Agent 1: Customer Support Agent (Main triage agent)
customer_support_agent = chat_client.as_agent(
    name="customer_support_agent",
    instructions=(
        "You are a friendly customer support agent for a travel company. "
        "Assess customer requests and route them to the appropriate specialist by "
        "calling the matching handoff tool: "
        "- For flight bookings or reservations: hand off to the booking agent. "
        "- For refunds, disputes, or billing issues: hand off to the disputes agent. "
        "- For trip confirmations or travel plan checks: hand off to the trip check agent. "
        "Be welcoming and ensure customers feel heard before routing them."
    ),
    require_per_service_call_history_persistence=True,
)


# Agent 2: Booking Agent (Flight booking specialist)
booking_agent = chat_client.as_agent(
    name="booking_agent",
    instructions=(
        "You are a flight booking specialist. Handle all flight reservations and bookings. "
        "When a customer wants to book a flight, collect their destination, travel dates, "
        "and confirm the booking. The flight is always confirmed regardless of destination. "
        "Reply with ONLY a JSON object (no prose, no code fences) using exactly these keys: "
        "destination, departure_date, return_date, booking_reference, passenger_name, "
        "flight_details, total_cost, status."
    ),
    require_per_service_call_history_persistence=True,
)

# Agent 3: Disputes Agent (Refund and billing specialist)
disputes_agent = chat_client.as_agent(
    name="disputes_agent",
    instructions=(
        "You are a disputes and refunds specialist. Handle customer complaints, refund "
        "requests, and billing disputes. Always approve refunds and process them back to the "
        "original payment method. "
        "Reply with ONLY a JSON object (no prose, no code fences) using exactly these keys: "
        "dispute_type, original_booking, refund_amount, refund_method, processing_time, "
        "reference_number, status."
    ),
    require_per_service_call_history_persistence=True,
)

# Agent 4: Trip Check Agent (Travel confirmation specialist)
trip_check_agent = chat_client.as_agent(
    name="trip_check_agent",
    instructions=(
        "You are a travel confirmation specialist. Verify and confirm customer travel plans, "
        "check itineraries, and provide travel status updates. Always confirm plans are in order. "
        "Reply with ONLY a JSON object (no prose, no code fences) using exactly these keys: "
        "trip_reference, destination, travel_dates, confirmation_status, special_notes, contact_info."
    ),
    require_per_service_call_history_persistence=True,
)





## အဆင့် ၄: Handoff Workflow ကို တည်ဆောက်ခြင်း

HandoffBuilder သည် ဖောက်သည်လိုအပ်ချက်အပေါ် အခြေခံပြီး ဖောက်သည်ပံ့ပိုးမှု အေးဂျင့်သည် အထူးပြုသူများထံ သွားရောက်ဆက်သွယ်နိုင်ရန် dynamic လုပ်သော workflow ကို ဖန်တီးပေးသည်။


In [ ]:
def build_workflow():
    """Build a fresh handoff workflow.

    Workflow runs are NOT isolated - state is preserved across calls to run(). We
    build a new instance per test so each scenario starts from a clean conversation.
    """
    return (
        HandoffBuilder(
            name="travel_support_handoff",
            participants=[customer_support_agent, booking_agent, disputes_agent, trip_check_agent],
            termination_condition=lambda conv: sum(1 for msg in conv if msg.role == "user") > 3,
        )
        .with_start_agent(customer_support_agent)  # Main agent that receives initial requests
        .add_handoff(customer_support_agent, [booking_agent, disputes_agent, trip_check_agent])
        .build()
    )


workflow = build_workflow()

display(HTML("""
<div style='padding: 20px; background: linear-gradient(135deg, #ff7043 0%, #ff5722 100%); color: white; border-radius: 8px; margin: 10px 0;'>
    <h3 style='margin: 0 0 15px 0;'>Handoff Workflow Built Successfully!</h3>
    <p style='margin: 0; line-height: 1.6;'>
        <strong>Handoff Flow:</strong><br>
        • User Request → <strong>Customer Support Agent</strong> (triage)<br>
        • Support Agent → <strong>Specialist Agent</strong> (dynamic handoff)<br>

        • Specialist → <strong>Resolution</strong> (expert handling)<br>

        • System → <strong>User Response</strong> (final result)
    </p>
</div>
"""))

## အဆင့် ၅: အဖြစ်အပျက်များကို ပြုလုပ်ရာတွင် အကူအညီပေးသော လုပ်ဆောင်ချက်များ

ဤလုပ်ဆောင်ချက်များသည် workflow အဖြစ်အပျက်များကို ကူညီ ပြုလုပ်ရန်နှင့် အသုံးပြုသူရဲ့ တောင်းဆိုချက်များကို ဟန်ဒါဖ်လုပ်ဆောင်ရာတွင် ကိုင်တွယ်ရန် ကူညီပေးသည်။


In [ ]:
async def drain_events(stream: AsyncIterable[WorkflowEvent]) -> list[WorkflowEvent]:
    """Collect all events from an async stream into a list."""
    return [event async for event in stream]


def _output_messages(data: Any) -> list[Message]:
    """Extract Message objects from an output event payload.

    Handoff output events carry an AgentResponse (has .messages), a single Message,
    or a list of Messages depending on the hop.
    """
    if hasattr(data, "messages"):
        return list(data.messages)
    if isinstance(data, Message):
        return [data]
    if isinstance(data, list):
        return [m for m in data if isinstance(m, Message)]
    return []


def handle_workflow_events(events: list[WorkflowEvent]) -> list[WorkflowEvent]:
    """Print progress and return pending user-input (request_info) events."""
    requests: list[WorkflowEvent] = []
    for event in events:
        if event.type == "handoff_sent":
            print(f"[Handoff: {event.data.source} -> {event.data.target}]")
        elif event.type == "status" and event.state in {
            WorkflowRunState.IDLE,
            WorkflowRunState.IDLE_WITH_PENDING_REQUESTS,
        }:
            print(f"[Workflow Status] {event.state}")
        elif event.type == "output":
            for message in _output_messages(event.data):
                if message.text.strip():
                    speaker = message.author_name or message.role
                    print(f"- {speaker}: {message.text}")
        elif event.type == "request_info" and isinstance(event.data, HandoffAgentUserRequest):
            requests.append(event)
    return requests


def collect_output_messages(events: list[WorkflowEvent]) -> list[Message]:
    """Gather every Message emitted as a workflow output across the given events."""
    messages: list[Message] = []
    for event in events:
        if event.type == "output":
            messages.extend(_output_messages(event.data))
    return messages


print("Helper functions defined for event processing")

## အဆင့် ၆: စမ်းသပ်မှုကိစ္စ ၁ - လေယာဉ်လက်မှတ်မှာယူခြင်း တောင်းဆိုမှု

ကျွန်တော်တို့ရဲ့ လက်သွားလက်ခံလုပ်ငန်းစဉ်ကို လေယာဉ်လက်မှတ်မှာယူခြင်း တောင်းဆိုမှုနဲ့ စမ်းသပ်ကြမယ်။ ဖောက်သည်ကူညီမှု_agent သည် မှာယူမှု_agent သို့ လက်သွား လက်ခံပေးရမည်။


In [ ]:
async def test_booking_handoff():
    """Test handoff workflow for flight booking requests."""

    display(HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Test Case 1: Flight Booking Request</h3>
        <p style='margin: 0;'><strong>Expected Flow:</strong> Customer Support → Booking Agent</p>
    </div>
    """))

    # Start the workflow with a fresh instance
    workflow = build_workflow()
    print("[User]: I want to book a flight to Paris for next month")
    all_events = await drain_events(
        workflow.run("I want to book a flight to Paris for next month", stream=True, include_status_events=True)
    )
    pending_requests = handle_workflow_events(all_events)

    # Handle any additional user input requests
    scripted_responses = [
        "I'd like to travel from New York to Paris on December 15th and return on December 22nd.",
        "Yes, please confirm the booking under the name John Smith."
    ]

    response_index = 0
    while pending_requests and response_index < len(scripted_responses):
        user_response = scripted_responses[response_index]
        print(f"\n[User]: {user_response}")

        responses = {
            req.request_id: HandoffAgentUserRequest.create_response(user_response)
            for req in pending_requests
        }
        new_events = await drain_events(
            workflow.run(stream=True, responses=responses, include_status_events=True)
        )
        all_events.extend(new_events)
        pending_requests = handle_workflow_events(new_events)

        response_index += 1

    # Extract and display the final booking result
    for message in collect_output_messages(all_events):
        if (message.author_name or "") == "booking_agent" and message.text.strip():
            try:
                booking_data = FlightBookingResult.model_validate_json(message.text)
                display_booking_result(booking_data)
                break
            except Exception as e:
                print(f"Could not parse booking result: {e}")


def display_booking_result(booking: FlightBookingResult):
    """Display flight booking result in a formatted section."""

    display(HTML(f"""
    <div style='padding: 20px; background: #e8f5e9; border-radius: 8px; margin: 15px 0; border-left: 4px solid #4caf50;'>
        <h3 style='margin: 0 0 15px 0; color: #2e7d32;'>✈️ Flight Booking Confirmed</h3>
        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 15px;'>
            <div>
                <strong style='color: #333;'>Booking Reference:</strong> {booking.booking_reference}<br>
                <strong style='color: #333;'>Passenger:</strong> {booking.passenger_name}<br>
                <strong style='color: #333;'>Status:</strong> <span style='color: #4caf50; font-weight: bold;'>{booking.status}</span>
            </div>
            <div>
                <strong style='color: #333;'>Destination:</strong> {booking.destination}<br>
                <strong style='color: #333;'>Total Cost:</strong> {booking.total_cost}<br>
                <strong style='color: #333;'>Departure:</strong> {booking.departure_date}
            </div>
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Flight Details:</strong> {booking.flight_details}
        </div>
        <div style='background: rgba(76,175,80,0.1); padding: 10px; border-radius: 4px; margin-top: 10px;'>
            <strong style='color: #2e7d32;'>✅ Success:</strong> Flight booking completed through handoff to booking specialist
        </div>
    </div>
    """))


await test_booking_handoff()
# Run the booking test

## အဆင့် ၇: စမ်းသပ်မှုအမှု ၂ - ရန်လုပ်ငန်း/ငွေပြန်ပေးရန်တောင်းဆိုမှု

ငွေပြန်ပေးရန်တောင်းဆိုမှုဖြင့် ကျွန်ုပ်တို့၏လက်လှမ်းပေးခြင်း အလုပ်စဉ်ကို စမ်းသပ်ကြည့်လိုက်ကြပါစို့။ ဖောက်သည်ဝန်ဆောင်မှု အေးဂျင့်သည် ရန်လုပ်ငန်း အေးဂျင့်ထံသို့ လက်လှမ်းပေးသင့်သည်။


In [ ]:
async def test_dispute_handoff():
    """Test handoff workflow for dispute/refund requests."""

    display(HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Test Case 2: Refund Request</h3>
        <p style='margin: 0;'><strong>Expected Flow:</strong> Customer Support → Disputes Agent</p>
    </div>
    """))

    # Start the workflow with a fresh instance
    workflow = build_workflow()
    print("[User]: I need to cancel my flight and get a refund")
    all_events = await drain_events(
        workflow.run("I need to cancel my flight and get a refund", stream=True, include_status_events=True)
    )
    pending_requests = handle_workflow_events(all_events)

    # Handle any additional user input requests
    scripted_responses = [
        "My booking reference is FL12345. I can't travel due to a family emergency.",
        "Yes, please process the full refund back to my credit card."
    ]

    response_index = 0
    while pending_requests and response_index < len(scripted_responses):
        user_response = scripted_responses[response_index]
        print(f"\n[User]: {user_response}")

        responses = {
            req.request_id: HandoffAgentUserRequest.create_response(user_response)
            for req in pending_requests
        }
        new_events = await drain_events(
            workflow.run(stream=True, responses=responses, include_status_events=True)
        )
        all_events.extend(new_events)
        pending_requests = handle_workflow_events(new_events)

        response_index += 1

    # Extract and display the final dispute result
    for message in collect_output_messages(all_events):
        if (message.author_name or "") == "disputes_agent" and message.text.strip():
            try:
                dispute_data = DisputeResult.model_validate_json(message.text)
                display_dispute_result(dispute_data)
                break
            except Exception as e:
                print(f"Could not parse dispute result: {e}")


def display_dispute_result(dispute: DisputeResult):
    """Display dispute resolution result in a formatted section."""

    display(HTML(f"""
    <div style='padding: 20px; background: #fff3e0; border-radius: 8px; margin: 15px 0; border-left: 4px solid #ff9800;'>
        <h3 style='margin: 0 0 15px 0; color: #f57c00;'>💰 Refund Processed</h3>
        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 15px;'>
            <div>
                <strong style='color: #333;'>Reference Number:</strong> {dispute.reference_number}<br>
                <strong style='color: #333;'>Dispute Type:</strong> {dispute.dispute_type}<br>
                <strong style='color: #333;'>Status:</strong> <span style='color: #ff9800; font-weight: bold;'>{dispute.status}</span>
            </div>
            <div>
                <strong style='color: #333;'>Refund Amount:</strong> {dispute.refund_amount}<br>
                <strong style='color: #333;'>Refund Method:</strong> {dispute.refund_method}<br>
                <strong style='color: #333;'>Processing Time:</strong> {dispute.processing_time}
            </div>
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Original Booking:</strong> {dispute.original_booking}
        </div>
        <div style='background: rgba(255,152,0,0.1); padding: 10px; border-radius: 4px; margin-top: 10px;'>
            <strong style='color: #f57c00;'>✅ Success:</strong> Refund processed through handoff to disputes specialist
        </div>
    </div>
    """))

await test_dispute_handoff()
    # Run the dispute test

## အဆင့် ၈ - စမ်းသပ်မှုကိစ္စ ၃ - ခရီးစစ်ဆေးခြင်း အတည်ပြုမေးခြင်း

ခရီးအတည်ပြုမေးခြင်းဖြင့် ကျွန်ုပ်တို့၏ လက်ဆောင်ပေးရေး လုပ်ငန်းစဉ်ကို စမ်းသပ်ကြမယ်။ ဖောက်သည်ထောက်ပံ့မှု အေးဂျင့်သည် ခရီးစစ်ဆေးအေးဂျင့်သို့ လက်ဆောင်ပေးရမည်ဖြစ်သည်။


In [ ]:
async def test_trip_check_handoff():
    """Test handoff workflow for trip confirmation requests."""

    display(HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Test Case 3: Trip Confirmation</h3>
        <p style='margin: 0;'><strong>Expected Flow:</strong> Customer Support → Trip Check Agent</p>
    </div>
    """))

    # Start the workflow with a fresh instance
    workflow = build_workflow()
    print("[User]: Can you confirm my travel plans are all set?")
    all_events = await drain_events(
        workflow.run("Can you confirm my travel plans are all set?", stream=True, include_status_events=True)
    )
    pending_requests = handle_workflow_events(all_events)

    # Handle any additional user input requests
    scripted_responses = [
        "I'm traveling to London next week. My confirmation number is TR98765.",
        "Perfect, thank you for checking everything is ready!"
    ]

    response_index = 0
    while pending_requests and response_index < len(scripted_responses):
        user_response = scripted_responses[response_index]
        print(f"\n[User]: {user_response}")

        responses = {
            req.request_id: HandoffAgentUserRequest.create_response(user_response)
            for req in pending_requests
        }
        new_events = await drain_events(
            workflow.run(stream=True, responses=responses, include_status_events=True)
        )
        all_events.extend(new_events)
        pending_requests = handle_workflow_events(new_events)

        response_index += 1
    # Extract and display the final trip check result
    for message in collect_output_messages(all_events):
        if (message.author_name or "") == "trip_check_agent" and message.text.strip():
            try:
                trip_data = TripCheckResult.model_validate_json(message.text)
                display_trip_check_result(trip_data)
                break
            except Exception as e:
                print(f"Could not parse trip check result: {e}")


def display_trip_check_result(trip: TripCheckResult):
    """Display trip confirmation result in a formatted section."""

    display(HTML(f"""
    <div style='padding: 20px; background: #f3e5f5; border-radius: 8px; margin: 15px 0; border-left: 4px solid #9c27b0;'>
        <h3 style='margin: 0 0 15px 0; color: #7b1fa2;'>🎯 Trip Confirmed</h3>
        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 15px;'>
            <div>
                <strong style='color: #333;'>Trip Reference:</strong> {trip.trip_reference}<br>
                <strong style='color: #333;'>Destination:</strong> {trip.destination}<br>
                <strong style='color: #333;'>Status:</strong> <span style='color: #9c27b0; font-weight: bold;'>{trip.confirmation_status}</span>
            </div>
            <div>
                <strong style='color: #333;'>Travel Dates:</strong> {trip.travel_dates}<br>
                <strong style='color: #333;'>Contact Info:</strong> {trip.contact_info}
            </div>
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Special Notes:</strong> {trip.special_notes}
        </div>
        <div style='background: rgba(156,39,176,0.1); padding: 10px; border-radius: 4px; margin-top: 10px;'>
            <strong style='color: #7b1fa2;'>✅ Success:</strong> Trip confirmed through handoff to trip check specialist
        </div>
    </div>
    """))


# Run the trip check test
await test_trip_check_handoff()

   

## အဆင့် ၉ - အလုပ်လည်ပတ်မှု ခေတ်နိုးချက် လေ့လာခြင်း - လက်လွှဲပေါက်စီးဆင်းမှု နားလည်ခြင်း


In [ ]:
async def analyze_handoff_patterns():
    """Analyze different handoff patterns and routing decisions."""

    display(HTML("""
    <div style='padding: 20px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #7b1fa2;'>Handoff Pattern Analysis</h3>
        <p style='margin: 0;'>Testing different request types to show routing decisions...</p>
    </div>
    """))

    test_requests = [
        "I want to book a round-trip flight to Tokyo",
        "I need a refund for my cancelled flight",
        "Please check if my travel itinerary is confirmed",
        "Can you help me with a billing dispute?"
    ]

    for i, request in enumerate(test_requests, 1):
        print(f"\n--- Test Request {i} ---")
        print(f"User: {request}")

        # Run workflow and capture routing decision
        # Run workflow (fresh instance) and capture the routing decision
        wf = build_workflow()
        events = await drain_events(wf.run(request, stream=True, include_status_events=True))

        # Analyze which agent was activated
        routed = False
        for message in collect_output_messages(events):
            name = message.author_name or message.role
            if name == "customer_support_agent" and message.text.strip():
                print(f"Support Agent: {message.text[:100]}...")
            elif name in ("booking_agent", "disputes_agent", "trip_check_agent"):
                agent_type = {
                    "booking_agent": "🛫 BOOKING SPECIALIST",
                    "disputes_agent": "💰 DISPUTES SPECIALIST",
                    "trip_check_agent": "🎯 TRIP CHECK SPECIALIST",
                }[name]
                print(f"Routed to: {agent_type}")
                routed = True
                break
        if not routed:
            print("(No specialist routing detected for this request.)")
    display(HTML("""
    <div style='padding: 25px; background: linear-gradient(135deg, #9c27b0 0%, #673ab7 100%); color: white; border-radius: 12px; 
                box-shadow: 0 4px 12px rgba(156,39,176,0.4); margin: 20px 0;'>
        <h2 style='margin: 0 0 20px 0;'>Handoff Analysis Results</h2>
        <div style='background: rgba(255,255,255,0.15); padding: 15px; border-radius: 8px;'>
            <h4 style='margin: 0 0 10px 0;'>Key Observations</h4>
            <ul style='margin: 0; padding-left: 20px; line-height: 1.6;'>
                <li><strong>Dynamic Routing:</strong> Customer support agent analyzes request intent</li>
                <li><strong>Context Preservation:</strong> Full conversation history maintained</li>
                <li><strong>Specialist Focus:</strong> Each agent handles their expertise area</li>
                <li><strong>Seamless Handoff:</strong> Users don't need to repeat information</li>
            </ul>
        </div>
    </div>
    """))

    # Run the analysis
await analyze_handoff_patterns()

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
